In [103]:
import torch
from jaxtyping import Float
import transformer_lens.utils as utils
from transformer_lens.hook_points import HookPoint
from transformer_lens import HookedTransformer 
import plotly.express as px

In [104]:
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [119]:
device = utils.get_device()
model = HookedTransformer.from_pretrained("pythia-14m", device=device)

Loaded pretrained model pythia-14m into HookedTransformer


In [113]:
text = "After John and Mary went to the store, Mary gave a bottle of milk to"

In [114]:
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    """
    Used for rendering 2D tensor as a plot.
    """

    px.imshow(
        utils.to_numpy(tensor), 
        labels={"x": xaxis, "y": yaxis}, 
        **kwargs
    ).show(renderer)

In [115]:
def print_name_shape_hook_function(activation: Float[torch.Tensor, "batch position head_index d_head"], hook: HookPoint):
    col_width = 30
    
    if hook.name.find("resid") != -1:
        colored_name = f"\033[1m{hook.name}\033[0m"
        padding = " " * max(0, col_width - len(hook.name))
        print(f"{colored_name}{padding} {activation.shape}")
    else:
        print(f"{hook.name:<{col_width}} {activation.shape}")

not_in_late_block_filter = lambda name: name.startswith("blocks.0.") or name.startswith("blocks.1.") or not name.startswith("blocks")

model.run_with_hooks(
    text,
    return_type=None,
    fwd_hooks=[(not_in_late_block_filter, print_name_shape_hook_function)]
)

hook_embed                     torch.Size([1, 17, 2048])
blocks.0.hook_resid_pre        torch.Size([1, 17, 2048])
blocks.0.ln1.hook_scale        torch.Size([1, 17, 1])
blocks.0.ln1.hook_normalized   torch.Size([1, 17, 2048])
blocks.0.ln1.hook_scale        torch.Size([1, 17, 1])
blocks.0.ln1.hook_normalized   torch.Size([1, 17, 2048])
blocks.0.ln1.hook_scale        torch.Size([1, 17, 1])
blocks.0.ln1.hook_normalized   torch.Size([1, 17, 2048])
blocks.0.attn.hook_q           torch.Size([1, 17, 8, 256])
blocks.0.attn.hook_k           torch.Size([1, 17, 8, 256])
blocks.0.attn.hook_v           torch.Size([1, 17, 8, 256])
blocks.0.attn.hook_rot_q       torch.Size([1, 17, 8, 256])
blocks.0.attn.hook_rot_k       torch.Size([1, 17, 8, 256])
blocks.0.attn.hook_attn_scores torch.Size([1, 8, 17, 17])
blocks.0.attn.hook_pattern     torch.Size([1, 8, 17, 17])
blocks.0.attn.hook_z           torch.Size([1, 17, 8, 256])
blocks.0.hook_attn_out         torch.Size([1, 17, 2048])
blocks.0.ln2.hook_scale   

In [116]:
str_tokens = model.to_str_tokens(text)
str_tokens

['<|endoftext|>',
 'After',
 ' John',
 ' and',
 ' Mary',
 ' went',
 ' to',
 ' the',
 ' store',
 ',',
 ' Mary',
 ' gave',
 ' a',
 ' bottle',
 ' of',
 ' milk',
 ' to']

In [118]:
# 1) Zapisujemy wartości aktywacji "przed" wejściem do bloku i "po" wejściu do bloku.
_, cache = model.run_with_cache(
    text,
    names_filter=lambda name: name.endswith("hook_resid_pre") or name.endswith("hook_resid_post")
)

# 2) Zbieramy aktywacje w tensory (n_layers, n_batches, n_positions, d_model)
pre_activations = cache.stack_activation("resid_pre")
post_activations = cache.stack_activation("resid_post")

# 3) Odejmujemy od wektorów post wektory pre (otrzymując f(x)) i bierzemy normę ||f(x)||.
# Usuwamy <|endoftext|>, bo dzieją się z nim dziwne rzeczy.
diff = (post_activations - pre_activations).flatten(start_dim=1, end_dim=-2).norm(dim=-1)[:, 1:]

print(f"pre={pre_activations.size()}")
print(f"post={post_activations.size()}")
print(f"diff={diff.size()}")

token_labels = [f"{token}_{index}" for index, token in enumerate(model.to_str_tokens(text)[1:])]
imshow(diff, x=token_labels, xaxis="Positions", yaxis="Layer", title="Change of activations between layers")

pre=torch.Size([16, 1, 17, 2048])
post=torch.Size([16, 1, 17, 2048])
diff=torch.Size([16, 16])
